
# 23A — V5 Discovery Fast-Track: E2 Freeze → Local Pairs → Nuisance Baseline

This notebook deliberately replaces the long sequence of separate E2 QA/freeze/coverage/pair/nuisance notebooks.

It performs **one consolidated transition** from the completed E2 batch research into a model-development-ready discovery target.

### What it does
1. Hard lineage/schema QA for E2 Batches 01–08.
2. Freeze the E2 research corpus exactly as collected.
3. Create:
   - `PRIMARY_TRUSTED`: high-confidence E2 subset for primary modeling.
   - `BROAD_SENSITIVITY`: high+medium E2 subset for robustness checks.
4. Append already-frozen Original DEV and E1 eligible corpora.
5. Collapse same-subject / same-axis / same-year / same-polarity rows to prevent pseudo-replication.
6. Generate all same-subject, opposite-polarity, 1–5 year local pairs.
7. Measure age/calendar/axis/wave nuisance predictability with **subject-grouped CV**.
8. Emit one readiness decision.

### What it does NOT do
- No astrology feature generation.
- No Production Control scoring.
- No CONFIRM research or loading.
- No event relabeling to manufacture pairs.
- No chronology balancing or pair cherry-picking.

**Important:** this is a transparent post-E2 discovery-protocol amendment.  
The original per-axis thresholds remain reported, but they are no longer blockers for exploratory DEV model development.  
The sealed CONFIRM set remains the final confirmatory test.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re, warnings
import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "SAJU_ML_V5_DISCOVERY_FASTTRACK_20260817"

def repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p] + list(p.parents):
        if (c / "saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run this notebook somewhere inside the Chartpalja repository.")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def as_bool(s):
    if isinstance(s, bool):
        return s
    return str(s).strip().lower() in {"true","1","yes","y"}

ROOT = repo_root()
CORPUS = ROOT / "research/ml_corpus/v5_ground_truth"

E2 = ROOT / "research/ml/artifacts/v5_dev_expansion_e2"
E2_RES = E2 / "results"
E2_WORKLIST = E2 / "V5_DEV_EXPANSION_E2_EVENT_WORKLIST_8BATCH.csv"
E2_FREEZE_DEC = E2 / "V5_DEV_EXPANSION_E2_FREEZE_DECISION.json"

ORIG_FREEZE = ROOT / "research/ml/artifacts/v5_event_freeze"
ORIG_ELIGIBLE = ORIG_FREEZE / "V5_DEV_EVENT_CORPUS_ELIGIBLE_FROZEN.csv"
ORIG_ROSTER = ROOT / "research/ml/artifacts/v5_identity_repair/V5_DEV_SUBJECT_ROSTER_160_REPAIRED.csv"

E1_FREEZE = ROOT / "research/ml/artifacts/v5_dev_expansion_e1_event_freeze"
E1_ELIGIBLE = E1_FREEZE / "V5_E1_EVENT_CORPUS_ELIGIBLE_FROZEN.csv"
E1_ROSTER = ROOT / "research/ml/artifacts/v5_dev_expansion_e1/V5_DEV_EXPANSION_E1_ROSTER_160.csv"

AMENDMENT = CORPUS / "V5_DISCOVERY_FASTTRACK_AMENDMENT.json"

OUT = ROOT / "research/ml/artifacts/v5_discovery_fasttrack"
OUT.mkdir(parents=True, exist_ok=True)

required = [E2_WORKLIST,E2_FREEZE_DEC,ORIG_ELIGIBLE,ORIG_ROSTER,E1_ELIGIBLE,E1_ROSTER,AMENDMENT]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

print("ROOT:", ROOT)
print("OUT :", OUT)


ROOT: /Users/sangjinlee/Desktop/projects/saju
OUT : /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v5_discovery_fasttrack


## 1. E2 hard QA and exact research-corpus freeze

In [2]:

# Load frozen E2 worklist/decision.
work = pd.read_csv(E2_WORKLIST)
dec = json.load(open(E2_FREEZE_DEC, encoding="utf-8"))
amend = json.load(open(AMENDMENT, encoding="utf-8"))

assert dec["status"] == "V5_DEV_EXPANSION_E2_ROSTER_FROZEN_READY_FOR_EVENT_COLLECTION"
assert len(work) == 160 and work.subject_id.nunique() == 160
assert work.preassigned_axis.value_counts().to_dict() == {"STATUS":70,"PROJECT":50,"COMPETITIVE":40}
assert dec["n_subjects"] == 160
assert dec["axis_counts"] == {"COMPETITIVE":40,"PROJECT":50,"STATUS":70}
assert dec["rules"]["confirm_remains_sealed"] is True
assert dec["rules"]["astrology_used_for_membership"] is False
assert dec["rules"]["control_used_for_membership"] is False

if "worklist_sha256" in dec:
    assert sha256_file(E2_WORKLIST) == dec["worklist_sha256"]

# Load all eight batch event files.
frames = []
for b in range(1,9):
    p = E2_RES / f"batch_{b:02d}" / f"V5_DEV_EXPANSION_E2_BATCH_{b:02d}_EVENTS.csv"
    if not p.exists():
        raise FileNotFoundError(p)
    x = pd.read_csv(p)
    x["source_batch_file"] = str(p.relative_to(ROOT))
    x["source_batch"] = b
    x["event_row_in_batch"] = np.arange(1, len(x)+1)
    frames.append(x)

e2_full = pd.concat(frames, ignore_index=True, sort=False)

required_cols = {
    "batch_id","subject_id","name","preassigned_axis","event_year","polarity",
    "event_type","event_description","source_url","source_quality","exclude"
}
missing = required_cols - set(e2_full.columns)
assert not missing, f"E2 event columns missing: {sorted(missing)}"

e2_full["exclude"] = e2_full["exclude"].map(as_bool)

# Every frozen subject must have at least one research/audit row.
assert set(e2_full.subject_id) == set(work.subject_id), (
    f"E2 subject coverage mismatch. events={e2_full.subject_id.nunique()} worklist={work.subject_id.nunique()}"
)

# Axis must match frozen roster.
axis_map = work.set_index("subject_id")["preassigned_axis"].to_dict()
bad_axis = e2_full[
    e2_full.apply(lambda r: axis_map.get(r.subject_id) != r.preassigned_axis, axis=1)
]
assert len(bad_axis) == 0, "E2 axis mismatch against frozen worklist."

# Retained research-eligible rows need core fields.
e2_research_eligible = e2_full[~e2_full.exclude].copy()
assert e2_research_eligible.polarity.isin(["positive","negative"]).all()
e2_research_eligible["event_year"] = pd.to_numeric(e2_research_eligible["event_year"], errors="raise").astype(int)
assert e2_research_eligible.event_year.between(1900, 2026).all()
assert e2_research_eligible.source_url.astype(str).str.startswith(("http://","https://")).all()

# No exact semantic duplicate rows.
dup_key = ["subject_id","preassigned_axis","event_year","polarity","event_type","event_description"]
dups = e2_research_eligible[e2_research_eligible.duplicated(dup_key, keep=False)].copy()
assert len(dups) == 0, f"Exact E2 semantic duplicates remain: {len(dups)} rows"

# Freeze exact collected E2 corpus. No factual/semantic edits are performed here.
full_path = OUT / "V5_FASTTRACK_E2_EVENT_CORPUS_FULL_FROZEN.csv"
elig_path = OUT / "V5_FASTTRACK_E2_EVENT_CORPUS_RESEARCH_ELIGIBLE_FROZEN.csv"
e2_full.to_csv(full_path, index=False)
e2_research_eligible.to_csv(elig_path, index=False)

print("E2 hard QA: PASS")
print("E2 subjects:", e2_full.subject_id.nunique())
print("E2 full rows:", len(e2_full))
print("E2 research-eligible rows:", len(e2_research_eligible))
print("E2 excluded/audit rows:", int(e2_full.exclude.sum()))


E2 hard QA: PASS
E2 subjects: 160
E2 full rows: 259
E2 research-eligible rows: 228
E2 excluded/audit rows: 31


## 2. Build primary trusted and broad sensitivity event subsets

In [3]:

# We do not alter the factual research corpus.
# Instead, modeling gets two deterministic confidence subsets.

# Explicitly fuzzy event-type tokens are excluded from PRIMARY_TRUSTED only.
# They remain in the frozen corpus and may remain in BROAD_SENSITIVITY.
FUZZY_PRIMARY_TOKENS = [
    "breakthrough",
    "critical_success",
    "landmark",
    "global_success",
    "professional_boxing_win",
    "quarterfinal_loss",
    "international_radio",
    "album_series",
    "no_qualifying_core_axis_event"
]

def fuzzy_primary(event_type):
    s = str(event_type).casefold()
    return any(tok in s for tok in FUZZY_PRIMARY_TOKENS)

e2_research_eligible["source_quality_norm"] = (
    e2_research_eligible.source_quality.astype(str).str.strip().str.lower()
)
e2_research_eligible["fuzzy_primary_semantics"] = e2_research_eligible.event_type.map(fuzzy_primary)

e2_primary = e2_research_eligible[
    (e2_research_eligible.source_quality_norm == "high")
    & (~e2_research_eligible.fuzzy_primary_semantics)
].copy()

e2_broad = e2_research_eligible[
    e2_research_eligible.source_quality_norm.isin(["high","medium"])
].copy()

primary_path = OUT / "V5_FASTTRACK_E2_EVENT_MODEL_PRIMARY_TRUSTED.csv"
broad_path = OUT / "V5_FASTTRACK_E2_EVENT_MODEL_BROAD_SENSITIVITY.csv"
e2_primary.to_csv(primary_path, index=False)
e2_broad.to_csv(broad_path, index=False)

quality_summary = (
    e2_research_eligible.groupby(["preassigned_axis","polarity","source_quality_norm"])
    .size().rename("rows").reset_index()
)
display(quality_summary)

print("PRIMARY_TRUSTED E2 rows:", len(e2_primary))
print("BROAD_SENSITIVITY E2 rows:", len(e2_broad))
print("Primary filtering is a modeling-confidence filter, not an event relabel/factual edit.")


,preassigned_axis,polarity,source_quality_norm,rows
0,COMPETITIVE,negative,high,13
1,COMPETITIVE,negative,medium,5
2,COMPETITIVE,positive,high,32
3,COMPETITIVE,positive,low,1
4,COMPETITIVE,positive,medium,8
5,PROJECT,negative,medium,2
6,PROJECT,positive,high,57
7,PROJECT,positive,medium,12
8,STATUS,negative,high,19
9,STATUS,negative,medium,4


PRIMARY_TRUSTED E2 rows: 185
BROAD_SENSITIVITY E2 rows: 227
Primary filtering is a modeling-confidence filter, not an event relabel/factual edit.


## 3. Append already-frozen Original DEV + E1, add birth/wave metadata

In [4]:

orig = pd.read_csv(ORIG_ELIGIBLE)
e1 = pd.read_csv(E1_ELIGIBLE)

orig_roster = pd.read_csv(ORIG_ROSTER)
e1_roster = pd.read_csv(E1_ROSTER)
e2_roster = work.copy()

# Already-frozen Original/E1 corpora were source/semantic frozen under prior notebooks,
# so all their eligible rows enter both primary and broad sets unchanged.
for x, wave in [(orig,"ORIGINAL"),(e1,"E1"),(e2_primary,"E2")]:
    x["collection_wave"] = wave
for x, wave in [(e2_broad,"E2")]:
    x["collection_wave"] = wave

orig["collection_wave"] = "ORIGINAL"
e1["collection_wave"] = "E1"

combined_primary_events = pd.concat([orig, e1, e2_primary], ignore_index=True, sort=False)
combined_broad_events = pd.concat([orig, e1, e2_broad], ignore_index=True, sort=False)

# Frozen roster metadata.
for r, wave in [(orig_roster,"ORIGINAL"),(e1_roster,"E1"),(e2_roster,"E2")]:
    r["collection_wave"] = wave

roster_cols = ["subject_id","name","preassigned_axis","birth_date","collection_wave"]
rosters = pd.concat([
    orig_roster[[c for c in roster_cols if c in orig_roster.columns]],
    e1_roster[[c for c in roster_cols if c in e1_roster.columns]],
    e2_roster[[c for c in roster_cols if c in e2_roster.columns]],
], ignore_index=True, sort=False)

assert rosters.subject_id.nunique() == 480, f"Expected 480 DEV identities, got {rosters.subject_id.nunique()}"
assert rosters.subject_id.duplicated().sum() == 0

rosters["birth_date"] = pd.to_datetime(rosters["birth_date"], errors="coerce")
assert rosters.birth_date.notna().all(), "Birth date missing in combined DEV roster."
rosters["birth_year"] = rosters.birth_date.dt.year.astype(int)

print("Combined DEV roster:", len(rosters))
print("Primary event rows:", len(combined_primary_events))
print("Broad event rows  :", len(combined_broad_events))


Combined DEV roster: 480
Primary event rows: 867
Broad event rows  : 909


## 4. Collapse same-year duplicates and generate deterministic 1–5y pairs

In [5]:

def normalize_events_for_pairing(events):
    x = events.copy()
    x["event_year"] = pd.to_numeric(x["event_year"], errors="raise").astype(int)
    assert x.polarity.isin(["positive","negative"]).all()

    # Collapse multiple same-polarity events in the same subject-year.
    # Annual astrology state is identical; multiple event rows would create pseudo-replicated pairs.
    agg = (
        x.groupby(
            ["subject_id","name","preassigned_axis","collection_wave","event_year","polarity"],
            as_index=False
        )
        .agg(
            event_count=("event_type","size"),
            event_types=("event_type", lambda s: " | ".join(sorted(set(map(str,s))))),
            source_urls=("source_url", lambda s: " | ".join(sorted(set(map(str,s)))) if "source_url" in x.columns else "")
        )
    )
    return agg

def build_local_pairs(events, roster):
    ev = normalize_events_for_pairing(events)
    birth = roster[["subject_id","birth_year"]].drop_duplicates()
    ev = ev.merge(birth, on="subject_id", how="left", validate="many_to_one")
    assert ev.birth_year.notna().all()

    out = []
    for (sid, axis, wave), g in ev.groupby(["subject_id","preassigned_axis","collection_wave"], sort=False):
        pos = g[g.polarity=="positive"]
        neg = g[g.polarity=="negative"]
        for _, p in pos.iterrows():
            for _, n in neg.iterrows():
                gap = abs(int(p.event_year)-int(n.event_year))
                if not (1 <= gap <= 5):
                    continue

                if p.event_year < n.event_year:
                    earlier, later = p, n
                    y = 1
                else:
                    earlier, later = n, p
                    y = 0

                out.append({
                    "subject_id": sid,
                    "name": earlier["name"],
                    "preassigned_axis": axis,
                    "collection_wave": wave,
                    "birth_year": int(earlier.birth_year),
                    "earlier_year": int(earlier.event_year),
                    "later_year": int(later.event_year),
                    "year_gap": int(gap),
                    "earlier_is_positive": int(y),
                    "earlier_polarity": earlier.polarity,
                    "later_polarity": later.polarity,
                    "earlier_event_count": int(earlier.event_count),
                    "later_event_count": int(later.event_count),
                    "earlier_event_types": earlier.event_types,
                    "later_event_types": later.event_types,
                })

    pairs = pd.DataFrame(out)
    if len(pairs):
        # Unique year-pair unit.
        key = ["subject_id","preassigned_axis","collection_wave","earlier_year","later_year","earlier_is_positive"]
        pairs = pairs.drop_duplicates(key).sort_values(
            ["collection_wave","subject_id","earlier_year","later_year"]
        ).reset_index(drop=True)
        counts = pairs.groupby("subject_id").size().rename("subject_pair_count")
        pairs = pairs.merge(counts, on="subject_id", how="left")
        pairs["subject_weight"] = 1.0 / pairs["subject_pair_count"]
        pairs["earlier_event_age"] = pairs["earlier_year"] - pairs["birth_year"]
        pairs["later_event_age"] = pairs["later_year"] - pairs["birth_year"]
        pairs["calendar_midpoint"] = (pairs["earlier_year"] + pairs["later_year"]) / 2.0
    return pairs

pairs_primary = build_local_pairs(combined_primary_events, rosters)
pairs_broad = build_local_pairs(combined_broad_events, rosters)

ppath = OUT / "V5_FASTTRACK_COMBINED_LOCAL_PAIRS_PRIMARY_TRUSTED.csv"
bpath = OUT / "V5_FASTTRACK_COMBINED_LOCAL_PAIRS_BROAD_SENSITIVITY.csv"
pairs_primary.to_csv(ppath, index=False)
pairs_broad.to_csv(bpath, index=False)

def coverage(pairs):
    if len(pairs)==0:
        return pd.DataFrame()
    z = pairs.groupby("preassigned_axis").agg(
        pair_rows=("subject_id","size"),
        pairable_subjects=("subject_id","nunique"),
        positive_earlier_share=("earlier_is_positive","mean")
    )
    total = pd.DataFrame({
        "pair_rows":[len(pairs)],
        "pairable_subjects":[pairs.subject_id.nunique()],
        "positive_earlier_share":[pairs.earlier_is_positive.mean()]
    }, index=["TOTAL"])
    return pd.concat([z,total])

print("PRIMARY")
display(coverage(pairs_primary))
print("BROAD")
display(coverage(pairs_broad))


PRIMARY


,pair_rows,pairable_subjects,positive_earlier_share
COMPETITIVE,134,57,0.582090
PROJECT,12,9,0.500000
STATUS,47,34,0.765957
TOTAL,193,100,0.621762


BROAD


,pair_rows,pairable_subjects,positive_earlier_share
COMPETITIVE,137,60,0.591241
PROJECT,13,10,0.538462
STATUS,47,34,0.765957
TOTAL,197,104,0.629442


## 5. Subject-grouped nuisance baselines (diagnostic, not a blocker)

In [6]:

def nuisance_cv(pairs, include_wave, dataset_name):
    if len(pairs)==0:
        return {"dataset":dataset_name,"include_wave":include_wave,"status":"NO_PAIRS"}

    try:
        from sklearn.compose import ColumnTransformer
        from sklearn.preprocessing import OneHotEncoder, StandardScaler
        from sklearn.pipeline import Pipeline
        from sklearn.linear_model import LogisticRegression
        from sklearn.model_selection import GroupKFold
        from sklearn.metrics import accuracy_score, balanced_accuracy_score
    except Exception as e:
        return {
            "dataset": dataset_name,
            "include_wave": include_wave,
            "status": "SKLEARN_UNAVAILABLE",
            "error": repr(e)
        }

    x = pairs.copy()
    y = x["earlier_is_positive"].astype(int).values
    groups = x["subject_id"].astype(str).values

    numeric = ["birth_year","earlier_event_age","calendar_midpoint","year_gap"]
    categorical = ["preassigned_axis"] + (["collection_wave"] if include_wave else [])
    features = numeric + categorical

    n_groups = x.subject_id.nunique()
    n_splits = min(5, n_groups)
    if n_splits < 2 or len(np.unique(y)) < 2:
        return {
            "dataset": dataset_name,
            "include_wave": include_wave,
            "status": "INSUFFICIENT_GROUPS_OR_CLASSES",
            "n_groups": int(n_groups),
            "class_counts": pd.Series(y).value_counts().to_dict()
        }

    pre = ColumnTransformer([
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
    ])
    clf = Pipeline([
        ("pre", pre),
        ("model", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            C=1.0
        ))
    ])

    accs, bals = [], []
    splitter = GroupKFold(n_splits=n_splits)
    for train_idx, test_idx in splitter.split(x[features], y, groups):
        clf.fit(x.iloc[train_idx][features], y[train_idx])
        pred = clf.predict(x.iloc[test_idx][features])
        accs.append(accuracy_score(y[test_idx], pred))
        bals.append(balanced_accuracy_score(y[test_idx], pred))

    return {
        "dataset": dataset_name,
        "include_wave": include_wave,
        "status": "OK",
        "n_pairs": int(len(x)),
        "n_subjects": int(n_groups),
        "positive_earlier_share": float(np.mean(y)),
        "group_cv_folds": int(n_splits),
        "mean_accuracy": float(np.mean(accs)),
        "mean_balanced_accuracy": float(np.mean(bals)),
        "fold_accuracy": [float(v) for v in accs],
        "fold_balanced_accuracy": [float(v) for v in bals]
    }

diagnostics = [
    nuisance_cv(pairs_primary, False, "PRIMARY_TRUSTED"),
    nuisance_cv(pairs_primary, True,  "PRIMARY_TRUSTED"),
    nuisance_cv(pairs_broad, False,   "BROAD_SENSITIVITY"),
    nuisance_cv(pairs_broad, True,    "BROAD_SENSITIVITY"),
]

diag_path = OUT / "V5_FASTTRACK_NUISANCE_DIAGNOSTICS.json"
json.dump(diagnostics, open(diag_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)

display(pd.DataFrame(diagnostics))


,dataset,include_wave,status,n_pairs,n_subjects,positive_earlier_share,group_cv_folds,mean_accuracy,mean_balanced_accuracy,fold_accuracy,fold_balanced_accuracy
0,PRIMARY_TRUSTED,False,OK,193,100,0.621762,5,0.595412,0.599226,"[0.5897435897435898, 0.5897435897435898, 0.692...","[0.5666666666666667, 0.6586206896551725, 0.690..."
1,PRIMARY_TRUSTED,True,OK,193,100,0.621762,5,0.595547,0.591689,"[0.6410256410256411, 0.5384615384615384, 0.666...","[0.6083333333333334, 0.5913793103448276, 0.664..."
2,BROAD_SENSITIVITY,False,OK,197,104,0.629442,5,0.578846,0.575523,"[0.6, 0.525, 0.5128205128205128, 0.61538461538...","[0.5866666666666667, 0.48860398860398857, 0.52..."
3,BROAD_SENSITIVITY,True,OK,197,104,0.629442,5,0.578974,0.579262,"[0.55, 0.55, 0.5384615384615384, 0.64102564102...","[0.5333333333333333, 0.5071225071225072, 0.577..."


## 6. Readiness decision

In [7]:

OLD_MIN = {"TOTAL":80,"COMPETITIVE":20,"PROJECT":20,"STATUS":25}

def pairable_counts(pairs):
    d = {"TOTAL": int(pairs.subject_id.nunique())}
    for axis in ["COMPETITIVE","PROJECT","STATUS"]:
        d[axis] = int(pairs.loc[pairs.preassigned_axis==axis,"subject_id"].nunique())
    return d

primary_counts = pairable_counts(pairs_primary)
broad_counts = pairable_counts(pairs_broad)

old_gate_pass = {k: primary_counts.get(k,0) >= v for k,v in OLD_MIN.items()}
warnings_list = []
for k,v in OLD_MIN.items():
    if not old_gate_pass[k]:
        warnings_list.append(
            f"Original predeclared coverage diagnostic below threshold: {k} "
            f"{primary_counts.get(k,0)} < {v}. Under the fast-track amendment this is a warning, not a discovery blocker."
        )

# True hard blockers retained in the amendment.
hard_blockers = []
if primary_counts["TOTAL"] < 30:
    hard_blockers.append(f"PRIMARY_TRUSTED has only {primary_counts['TOTAL']} pairable subjects (<30).")
if len(pairs_primary) == 0 or pairs_primary.earlier_is_positive.nunique() < 2:
    hard_blockers.append("PRIMARY_TRUSTED does not contain both target classes.")

# No CONFIRM paths are loaded anywhere in this notebook.
confirm_loaded = False

if hard_blockers:
    status = "V5_DISCOVERY_FASTTRACK_BLOCKED_HARD_DATA_FAILURE"
    astrology_allowed = False
else:
    status = "V5_DISCOVERY_FASTTRACK_READY_FOR_ASTROLOGY_AND_MODEL_DEVELOPMENT"
    astrology_allowed = True

decision = {
    "version": "V5_DISCOVERY_FASTTRACK_READINESS_DECISION_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": status,
    "methodology_mode": "POST_OBSERVATION_EXPLORATORY_DEV_WITH_SEALED_CONFIRM",
    "E2": {
        "subjects_n": int(e2_full.subject_id.nunique()),
        "full_rows_n": int(len(e2_full)),
        "research_eligible_rows_n": int(len(e2_research_eligible)),
        "primary_trusted_rows_n": int(len(e2_primary)),
        "broad_sensitivity_rows_n": int(len(e2_broad)),
        "full_frozen_sha256": sha256_file(full_path),
        "research_eligible_frozen_sha256": sha256_file(elig_path),
        "primary_trusted_sha256": sha256_file(primary_path),
        "broad_sensitivity_sha256": sha256_file(broad_path),
    },
    "combined_primary": {
        "pair_rows_n": int(len(pairs_primary)),
        "pairable_subjects": primary_counts,
        "positive_earlier_share": float(pairs_primary.earlier_is_positive.mean()) if len(pairs_primary) else None,
        "pairs_sha256": sha256_file(ppath)
    },
    "combined_broad": {
        "pair_rows_n": int(len(pairs_broad)),
        "pairable_subjects": broad_counts,
        "positive_earlier_share": float(pairs_broad.earlier_is_positive.mean()) if len(pairs_broad) else None,
        "pairs_sha256": sha256_file(bpath)
    },
    "old_predeclared_thresholds_reported_only": OLD_MIN,
    "old_gate_primary_pass_by_dimension": old_gate_pass,
    "warnings": warnings_list,
    "hard_blockers": hard_blockers,
    "nuisance_diagnostics_sha256": sha256_file(diag_path),
    "amendment_sha256": sha256_file(AMENDMENT),
    "rules": {
        "E3_required_for_discovery": False,
        "per_axis_minima_are_hard_blockers": False,
        "nuisance_accuracy_is_hard_blocker": False,
        "nuisance_must_be_outperformed_by_astrology_candidate": True,
        "same_year_pseudo_replication_collapsed": True,
        "event_relabeling_for_pairability": False,
        "astrology_generated": False,
        "control_scored": False,
        "confirm_loaded_or_researched": confirm_loaded,
        "confirm_remains_sealed": True
    },
    "astrology_generation_allowed": astrology_allowed,
    "control_scoring_allowed": False,
    "confirm_event_research_allowed": False,
    "next_rule": (
        "If READY: generate astrology primitives only for the frozen paired subject-years, "
        "run a shared-coefficient subject-grouped V5 discovery tournament, compare astrology-only "
        "and nuisance+astrology against nuisance-only, run PRIMARY/BROAD/wave sensitivity, "
        "then freeze one candidate architecture before opening CONFIRM."
    )
}

decision_path = OUT / "V5_FASTTRACK_DEV_READINESS_DECISION.json"
json.dump(decision, open(decision_path,"w",encoding="utf-8"), ensure_ascii=False, indent=2)

summary = pd.DataFrame([
    {"dataset":"PRIMARY_TRUSTED", **primary_counts, "pair_rows":len(pairs_primary)},
    {"dataset":"BROAD_SENSITIVITY", **broad_counts, "pair_rows":len(pairs_broad)}
])
summary_path = OUT / "V5_FASTTRACK_PAIR_COVERAGE_SUMMARY.csv"
summary.to_csv(summary_path,index=False)

print(json.dumps(decision, ensure_ascii=False, indent=2))
print("\nSend back these 4 files:")
print("1.", decision_path.name)
print("2.", ppath.name)
print("3.", bpath.name)
print("4.", diag_path.name)


{
  "version": "V5_DISCOVERY_FASTTRACK_READINESS_DECISION_V1",
  "notebook_version": "SAJU_ML_V5_DISCOVERY_FASTTRACK_20260817",
  "created_at": "2026-08-17T12:46:18",
  "status": "V5_DISCOVERY_FASTTRACK_READY_FOR_ASTROLOGY_AND_MODEL_DEVELOPMENT",
  "methodology_mode": "POST_OBSERVATION_EXPLORATORY_DEV_WITH_SEALED_CONFIRM",
  "E2": {
    "subjects_n": 160,
    "full_rows_n": 259,
    "research_eligible_rows_n": 228,
    "primary_trusted_rows_n": 185,
    "broad_sensitivity_rows_n": 227,
    "full_frozen_sha256": "09fc3f5f81dec95973910703610ae70aef4e14dfe98b2cc438d599a0f733d3af",
    "research_eligible_frozen_sha256": "c8964eb73cefab51bd66dbedefca85f654740e798f46a1eb3bb7f5185bc0adbf",
    "primary_trusted_sha256": "6588d878b3f01d926e61c5ecf0f79b6592508bb4f5ae6a65dd7e2c11a06868a0",
    "broad_sensitivity_sha256": "510ed0f1c003b746d9438667df359b554a51ac2237a77de555011210be9d315a"
  },
  "combined_primary": {
    "pair_rows_n": 193,
    "pairable_subjects": {
      "TOTAL": 100,
      "COMP


## After Run All

Expected successful status:

```text
V5_DISCOVERY_FASTTRACK_READY_FOR_ASTROLOGY_AND_MODEL_DEVELOPMENT
```

Send back **only these four files**:

```text
V5_FASTTRACK_DEV_READINESS_DECISION.json
V5_FASTTRACK_COMBINED_LOCAL_PAIRS_PRIMARY_TRUSTED.csv
V5_FASTTRACK_COMBINED_LOCAL_PAIRS_BROAD_SENSITIVITY.csv
V5_FASTTRACK_NUISANCE_DIAGNOSTICS.json
```

If the notebook raises a hard-data error, send the error + `V5_FASTTRACK_DEV_READINESS_DECISION.json` if one was written.

Do **not** research CONFIRM and do **not** score Production Control yet.
